# Advanced packages - LAK (Lake)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `mf6_notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`mf6-adv-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-adv-maw`) - wells (WEL) -> Multi-Aquifer Well
- **LAK** (`mf6-adv-lak`) - high-K lake -> Lake package *(this notebook)*
- **SFR** (`mf6-adv-sfr`) - river (RIV) -> Streamflow Routing
- **MVR** (`mf6-adv-mvr`) - Water Mover (requires UZF, LAK, SFR)
- **Processing output** (`mf6-adv-processing`) - run the model and plot results

Replace the high-conductivity cells that approximate the lake with the Lake (LAK) package, which simulates lake stage and its exchange with the groundwater system.

## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
import numpy as np
from mf6_notebook_helpers import (
    IN2FT,
    drop_packages,
    load_or_create_advanced_model,
    load_spatial_data,
    load_temporal_data,
)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load or create the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use), then read the grid shape and the lake location and forcing data.

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)
nper = sim.tdis.nper.array
shape3d = (gwf.dis.nlay.array, gwf.dis.nrow.array, gwf.dis.ncol.array)

In [ ]:
temporal_df = load_temporal_data(sample_frequency)
_, lake_location, lake_area = load_spatial_data()

## Build the LAK package

Use `get_lak_connections` to build the lake connections from the lake footprint (and the idomain that deactivates the high-K lake cells), then add the lake with rainfall and evaporation and lake stage/exchange observations.

In [ ]:
# lake_array: 0 in the lake footprint (lake number 0), -1 elsewhere
lake_array = np.full(shape3d, -1, dtype=int)
lake_array[0, :, :][lake_location == 1] = 0

idomain, lake_connection_dict, lake_connection_data = (
    flopy.mf6.utils.get_lak_connections(
        gwf.modelgrid,
        lake_array,
        bedleak=1e-2,
    )
)

In [ ]:
# LAK package data: <ifno> <strt> <nlakeconn> <boundname>
lake_pakagedata = [
    (int(key), 13.0, value, "lake_harbaugh")
    for key, value in lake_connection_dict.items()
]
nlakes = len(lake_pakagedata)

In [ ]:
lake_obs = {
    f"{name}.lake.obs.csv": [
        ("LAKE-STAGE", "STAGE", "LAKE_HARBAUGH"),
        ("LAKE-SWGW", "LAK", "LAKE_HARBAUGH"),
    ]
}

In [ ]:
# lake stress period data with rainfall and evaporation
lake_spd = {}
rain_tag = "PRCP (Inches)"
pet_tag = "lake et (inches)"
for n in range(nper):
    row = temporal_df.iloc[n]
    rain = float(row[rain_tag]) * IN2FT
    pet = float(row[pet_tag]) * IN2FT
    lake_spd[n] = [(0, "rainfall", rain), (0, "evaporation", pet)]

In [ ]:
# apply the LAK idomain (deactivates the high-K lake cells) and add the package,
# replacing the EVT package (idempotent: drop any prior LAK/EVT first)
gwf.dis.idomain = idomain
drop_packages(gwf, "LAK-1", "EVT_0")
lak = flopy.mf6.ModflowGwflak(
    gwf,
    boundnames=True,
    print_input=True,
    print_stage=True,
    mover=True,
    pname="LAK-1",
    stage_filerecord=f"{name}.lak.hds",
    nlakes=nlakes,
    packagedata=lake_pakagedata,
    connectiondata=lake_connection_data,
    perioddata=lake_spd,
)
lak.obs.initialize(filename=f"{name}.lak.obs", print_input=True, continuous=lake_obs)

In [ ]:
# rebuild the GWF observation package without the (now redundant) lake head obs
pak = gwf.get_package("GWF-OBS")
if pak is not None:
    gwf_obs = {}
    for key, value in pak.continuous.get_active_key_list():
        if key != f"{name}.lake.obs.csv":
            gwf_obs[key] = value.get_data().tolist()
    drop_packages(gwf, "GWF-OBS")
    flopy.mf6.ModflowUtlobs(
        gwf,
        print_input=True,
        continuous=gwf_obs,
    )
gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the LAK package is valid.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print("LAK advanced model ran successfully")

## Recap

In this notebook you replaced the high-conductivity lake cells and the Evapotranspiration (**EVT**) package with the Lake (**LAK**) package, which simulates lake stage and its exchange with the groundwater system, then wrote the updated advanced model back to `models/` and ran it.